# Advice Metrics Analysis: LLM vs Human Advice

Compares computational linguistic metrics between LLM-generated and human advice.
All metrics are descriptive (no LLM-as-judge). Operationalizes constructs from:
- Appraisal theory (Engagement, Attitude, Graduation)
- Advice-giving style (directiveness, hedging, relationship lexicons)

**Prerequisites:** Run `scripts/analysis/compute_advice_metrics.py --from-checkpoints`

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from scipy import stats

# Publication settings
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 11,
    'axes.labelsize': 11,
    'axes.titlesize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 9,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
})

# Paths
DATA_DIR = Path('../data/advice_metrics')
FIGURES_DIR = Path('../figures')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
# Load latest metrics file
metrics_files = sorted(DATA_DIR.glob('advice_metrics_*.parquet'))
if not metrics_files:
    raise FileNotFoundError(
        "No metrics file found. Run: python scripts/analysis/compute_advice_metrics.py --from-checkpoints"
    )

df = pd.read_parquet(metrics_files[-1])
print(f"Loaded: {metrics_files[-1].name}")
print(f"Shape: {df.shape}")
print(f"\nSources:")
print(df['source'].value_counts().sort_index())

FileNotFoundError: No metrics file found. Run: python scripts/analysis/compute_advice_metrics.py --from-checkpoints

In [ ]:
# Source display names and styling
SOURCE_LABELS = {
    'human': 'Human',
    'gemini': 'Gemini 2.5 FL',
    'deepseek': 'DeepSeek v3',
    'ministral': 'Ministral 8B',
    'gpt_nano': 'GPT-4.1-nano',
}

# Greyscale palette (human = black, LLMs = greys)
SOURCE_COLORS = {
    'human': '#111111',
    'gemini': '#444444',
    'deepseek': '#777777',
    'ministral': '#aaaaaa',
    'gpt_nano': '#cccccc',
}

# Order: human first, then LLMs
SOURCE_ORDER = ['human', 'gemini', 'deepseek', 'ministral', 'gpt_nano']
# Filter to sources actually present
SOURCE_ORDER = [s for s in SOURCE_ORDER if s in df['source'].unique()]

print(f"Sources in order: {SOURCE_ORDER}")

## Summary Statistics

In [ ]:
# Key metrics summary
summary_cols = [
    'n_tokens', 'avg_sent_len', 'n_paragraphs',
    'certainty_ratio', 'modal_ratio',
    'imperative_count', 'you_density', 'question_ratio',
    'first_person_ratio', 'second_person_ratio',
    'leave_ratio', 'redflag_count', 'therapy_count',
    'intensification_ratio',
    'sentiment_compound',
]

summary = df.groupby('source')[summary_cols].agg(['median', 'mean', 'std'])
# Reorder
summary = summary.loc[[s for s in SOURCE_ORDER if s in summary.index]]
summary.round(3)

## Figure 1: Structural & Engagement Metrics

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8), dpi=150)

# Metrics to plot in this figure
panel_metrics = [
    ('n_tokens', 'Response Length (tokens)'),
    ('avg_sent_len', 'Avg. Sentence Length (tokens)'),
    ('certainty_ratio', 'Certainty Ratio\n(boosters / [boosters+hedges])'),
    ('modal_ratio', 'Modal Ratio\n(deontic / [deontic+epistemic])'),
    ('question_ratio', 'Question Ratio\n(questions / sentences)'),
    ('you_density', 'You-Density\n(2nd person / sentences)'),
]

panel_labels = ['(a)', '(b)', '(c)', '(d)', '(e)', '(f)']

for idx, (metric, ylabel) in enumerate(panel_metrics):
    ax = axes.flat[idx]
    
    # Prepare data for box plot
    data_groups = []
    positions = []
    for i, source in enumerate(SOURCE_ORDER):
        vals = df[df['source'] == source][metric].dropna()
        data_groups.append(vals)
        positions.append(i)
    
    bp = ax.boxplot(
        data_groups, positions=positions, widths=0.6,
        patch_artist=True, showfliers=False,
        medianprops=dict(color='black', linewidth=1.5),
        whiskerprops=dict(color='#333333'),
        capprops=dict(color='#333333'),
    )
    
    for patch, source in zip(bp['boxes'], SOURCE_ORDER):
        patch.set_facecolor(SOURCE_COLORS.get(source, '#888888'))
        patch.set_edgecolor('#222222')
        patch.set_linewidth(0.5)
    
    ax.set_xticks(positions)
    ax.set_xticklabels([SOURCE_LABELS.get(s, s) for s in SOURCE_ORDER], 
                        rotation=30, ha='right', fontsize=8)
    ax.set_ylabel(ylabel)
    ax.set_title(panel_labels[idx], fontweight='bold', loc='left')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='y', alpha=0.3, linestyle='--')

plt.tight_layout()
plt.show()

In [ ]:
# Save Figure 1
fig.savefig(FIGURES_DIR / 'fig_advice_structure_engagement.pdf', bbox_inches='tight', dpi=300)
fig.savefig(FIGURES_DIR / 'fig_advice_structure_engagement.png', bbox_inches='tight', dpi=300)
print("Saved: fig_advice_structure_engagement.pdf/png")

## Figure 2: Directiveness & Relationship Lexicons

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8), dpi=150)

panel_metrics = [
    ('imperative_count', 'Imperative Sentences'),
    ('conditional_count', 'Conditional Clauses'),
    ('leave_ratio', 'Leave Ratio\n(leave / [leave+stay])'),
    ('redflag_count', 'Red Flag Language'),
    ('therapy_count', 'Therapy/Self-care Language'),
    ('sentiment_compound', 'Sentiment (VADER compound)'),
]

panel_labels = ['(a)', '(b)', '(c)', '(d)', '(e)', '(f)']

for idx, (metric, ylabel) in enumerate(panel_metrics):
    ax = axes.flat[idx]
    
    data_groups = []
    for source in SOURCE_ORDER:
        vals = df[df['source'] == source][metric].dropna()
        data_groups.append(vals)
    
    bp = ax.boxplot(
        data_groups, positions=range(len(SOURCE_ORDER)), widths=0.6,
        patch_artist=True, showfliers=False,
        medianprops=dict(color='black', linewidth=1.5),
        whiskerprops=dict(color='#333333'),
        capprops=dict(color='#333333'),
    )
    
    for patch, source in zip(bp['boxes'], SOURCE_ORDER):
        patch.set_facecolor(SOURCE_COLORS.get(source, '#888888'))
        patch.set_edgecolor('#222222')
        patch.set_linewidth(0.5)
    
    ax.set_xticks(range(len(SOURCE_ORDER)))
    ax.set_xticklabels([SOURCE_LABELS.get(s, s) for s in SOURCE_ORDER],
                        rotation=30, ha='right', fontsize=8)
    ax.set_ylabel(ylabel)
    ax.set_title(panel_labels[idx], fontweight='bold', loc='left')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='y', alpha=0.3, linestyle='--')

plt.tight_layout()
plt.show()

In [ ]:
# Save Figure 2
fig.savefig(FIGURES_DIR / 'fig_advice_directiveness_lexicons.pdf', bbox_inches='tight', dpi=300)
fig.savefig(FIGURES_DIR / 'fig_advice_directiveness_lexicons.png', bbox_inches='tight', dpi=300)
print("Saved: fig_advice_directiveness_lexicons.pdf/png")

## Figure 3: LLM vs Human Divergence (Effect Sizes)

Cohen's d for each metric comparing each LLM to human baseline.

In [ ]:
def cohens_d(group1, group2):
    """Compute Cohen's d (positive = group1 > group2)."""
    n1, n2 = len(group1), len(group2)
    if n1 < 2 or n2 < 2:
        return 0.0
    var1, var2 = group1.var(), group2.var()
    pooled_std = np.sqrt(((n1 - 1) * var1 + (n2 - 1) * var2) / (n1 + n2 - 2))
    if pooled_std == 0:
        return 0.0
    return (group1.mean() - group2.mean()) / pooled_std


# Metrics to compare
effect_metrics = [
    ('n_tokens', 'Response Length'),
    ('avg_sent_len', 'Sentence Length'),
    ('certainty_ratio', 'Certainty'),
    ('modal_ratio', 'Deontic Modality'),
    ('imperative_count', 'Imperatives'),
    ('you_density', 'You-Density'),
    ('question_ratio', 'Questions'),
    ('conditional_count', 'Conditionals'),
    ('leave_ratio', 'Leave Orientation'),
    ('redflag_count', 'Red Flag Language'),
    ('therapy_count', 'Therapy Language'),
    ('intensification_ratio', 'Intensification'),
    ('sentiment_compound', 'Sentiment'),
    ('first_person_ratio', '1st Person'),
    ('second_person_ratio', '2nd Person'),
]

human_data = df[df['source'] == 'human']
llm_sources = [s for s in SOURCE_ORDER if s != 'human']

# Compute effect sizes
effect_sizes = []
for metric, label in effect_metrics:
    human_vals = human_data[metric].dropna()
    for source in llm_sources:
        llm_vals = df[df['source'] == source][metric].dropna()
        d = cohens_d(llm_vals, human_vals)
        effect_sizes.append({
            'metric': label,
            'source': source,
            'd': d,
        })

effects_df = pd.DataFrame(effect_sizes)
effects_pivot = effects_df.pivot(index='metric', columns='source', values='d')
effects_pivot = effects_pivot[llm_sources]  # Ensure order

# Reorder metrics by average absolute effect
metric_order = effects_pivot.abs().mean(axis=1).sort_values(ascending=True).index
effects_pivot = effects_pivot.loc[metric_order]

print("Effect sizes (Cohen's d, LLM vs Human):")
print(effects_pivot.round(3).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8), dpi=150)

y_positions = np.arange(len(effects_pivot))
n_sources = len(llm_sources)
bar_height = 0.8 / n_sources

for i, source in enumerate(llm_sources):
    offset = (i - n_sources / 2 + 0.5) * bar_height
    ax.barh(
        y_positions + offset,
        effects_pivot[source],
        height=bar_height,
        color=SOURCE_COLORS.get(source, '#888888'),
        edgecolor='#222222',
        linewidth=0.5,
        label=SOURCE_LABELS.get(source, source),
    )

# Reference lines
ax.axvline(x=0, color='#222222', linewidth=0.8)
for threshold in [-0.8, -0.5, -0.2, 0.2, 0.5, 0.8]:
    ax.axvline(x=threshold, color='#cccccc', linewidth=0.5, linestyle=':')

ax.set_yticks(y_positions)
ax.set_yticklabels(effects_pivot.index)
ax.set_xlabel("Cohen's $d$ (LLM $-$ Human)", fontweight='bold')
ax.legend(loc='lower right', framealpha=0.9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='x', alpha=0.3, linestyle='--')

# Effect size labels
ax.text(0.2, -1.2, 'small', fontsize=8, ha='center', color='#666666', style='italic')
ax.text(0.5, -1.2, 'medium', fontsize=8, ha='center', color='#666666', style='italic')
ax.text(0.8, -1.2, 'large', fontsize=8, ha='center', color='#666666', style='italic')

plt.tight_layout()
plt.show()

In [ ]:
# Save Figure 3
fig.savefig(FIGURES_DIR / 'fig_advice_effect_sizes.pdf', bbox_inches='tight', dpi=300)
fig.savefig(FIGURES_DIR / 'fig_advice_effect_sizes.png', bbox_inches='tight', dpi=300)
print("Saved: fig_advice_effect_sizes.pdf/png")

## Figure 4: Pronoun Profiles

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5), dpi=150)

pronoun_cols = ['first_person_ratio', 'second_person_ratio', 'third_person_ratio']
pronoun_labels = ['1st Person\n(I, me, my)', '2nd Person\n(you, your)', '3rd Person\n(he, she, they)']

x = np.arange(len(pronoun_labels))
width = 0.8 / len(SOURCE_ORDER)

for i, source in enumerate(SOURCE_ORDER):
    source_data = df[df['source'] == source]
    means = [source_data[col].mean() for col in pronoun_cols]
    sems = [source_data[col].sem() for col in pronoun_cols]
    offset = (i - len(SOURCE_ORDER) / 2 + 0.5) * width
    ax.bar(
        x + offset, means, width,
        yerr=sems,
        color=SOURCE_COLORS.get(source, '#888888'),
        edgecolor='#222222',
        linewidth=0.5,
        label=SOURCE_LABELS.get(source, source),
        capsize=2,
        error_kw={'lw': 0.8, 'capthick': 0.8},
    )

ax.set_xticks(x)
ax.set_xticklabels(pronoun_labels)
ax.set_ylabel('Proportion of All Pronouns')
ax.legend(loc='upper right', framealpha=0.9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3, linestyle='--')

plt.tight_layout()
plt.show()

In [ ]:
# Save Figure 4
fig.savefig(FIGURES_DIR / 'fig_advice_pronouns.pdf', bbox_inches='tight', dpi=300)
fig.savefig(FIGURES_DIR / 'fig_advice_pronouns.png', bbox_inches='tight', dpi=300)
print("Saved: fig_advice_pronouns.pdf/png")

## Statistical Tests

Kruskal-Wallis H-test (non-parametric) for each metric across sources,
followed by pairwise Mann-Whitney U for LLM vs Human comparisons.

In [ ]:
from scipy.stats import kruskal, mannwhitneyu

test_metrics = [
    'n_tokens', 'avg_sent_len', 'certainty_ratio', 'modal_ratio',
    'imperative_count', 'you_density', 'question_ratio',
    'leave_ratio', 'redflag_count', 'therapy_count',
    'intensification_ratio', 'sentiment_compound',
    'first_person_ratio', 'second_person_ratio',
]

# Kruskal-Wallis across all sources
print("Kruskal-Wallis H-test (all sources):")
print(f"{'Metric':<25} {'H':>8} {'p':>12}")
print("-" * 50)

for metric in test_metrics:
    groups = [df[df['source'] == s][metric].dropna() for s in SOURCE_ORDER]
    groups = [g for g in groups if len(g) > 0]
    if len(groups) >= 2:
        h_stat, p_val = kruskal(*groups)
        sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else ''
        print(f"{metric:<25} {h_stat:>8.1f} {p_val:>12.2e} {sig}")

In [ ]:
# Pairwise: each LLM vs Human
print("\nMann-Whitney U: LLM vs Human (two-sided):")
print(f"{'Metric':<25} {'Source':<15} {'U':>10} {'p':>12} {'d':>7}")
print("-" * 75)

pairwise_results = []
for metric in test_metrics:
    human_vals = human_data[metric].dropna()
    if len(human_vals) < 10:
        continue
    for source in llm_sources:
        llm_vals = df[df['source'] == source][metric].dropna()
        if len(llm_vals) < 10:
            continue
        u_stat, p_val = mannwhitneyu(llm_vals, human_vals, alternative='two-sided')
        d = cohens_d(llm_vals, human_vals)
        sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else ''
        print(f"{metric:<25} {source:<15} {u_stat:>10.0f} {p_val:>12.2e} {d:>+7.3f} {sig}")
        pairwise_results.append({
            'metric': metric, 'source': source,
            'U': u_stat, 'p': p_val, 'd': d,
        })

pairwise_df = pd.DataFrame(pairwise_results)

## Universal vs Provider-Specific Patterns

Where do ALL LLMs diverge from humans similarly (universal LLM bias)
vs where do they diverge from each other (provider-specific)?

In [ ]:
# Classify patterns
if len(pairwise_df) > 0:
    # For each metric, check if all LLMs go the same direction relative to human
    print("UNIVERSAL PATTERNS (all LLMs diverge from human in same direction, |d| > 0.2):")
    print("=" * 70)
    
    for metric in test_metrics:
        metric_effects = pairwise_df[pairwise_df['metric'] == metric]
        if len(metric_effects) == 0:
            continue
        
        ds = metric_effects['d'].values
        all_positive = all(d > 0.2 for d in ds)
        all_negative = all(d < -0.2 for d in ds)
        
        if all_positive or all_negative:
            direction = "LLM > Human" if all_positive else "LLM < Human"
            avg_d = ds.mean()
            print(f"  {metric:<25} {direction:<15} avg d = {avg_d:+.3f}")
    
    print("\nPROVIDER-SPECIFIC PATTERNS (LLMs disagree, max spread > 0.3):")
    print("=" * 70)
    
    for metric in test_metrics:
        metric_effects = pairwise_df[pairwise_df['metric'] == metric]
        if len(metric_effects) == 0:
            continue
        
        ds = metric_effects['d'].values
        spread = ds.max() - ds.min()
        
        if spread > 0.3:
            details = ', '.join(
                f"{row['source']}={row['d']:+.2f}" 
                for _, row in metric_effects.iterrows()
            )
            print(f"  {metric:<25} spread={spread:.2f}  [{details}]")

## Per-Metric Normalized Comparison (Radar-style as Grouped Bars)

In [ ]:
# Normalized medians (z-scored across sources) for compact comparison
norm_metrics = [
    'certainty_ratio', 'modal_ratio', 'you_density', 'question_ratio',
    'leave_ratio', 'therapy_count', 'intensification_ratio', 'sentiment_compound',
]

medians = df.groupby('source')[norm_metrics].median()
medians = medians.loc[[s for s in SOURCE_ORDER if s in medians.index]]

# Z-score across sources
z_medians = (medians - medians.mean()) / medians.std()

fig, ax = plt.subplots(figsize=(12, 5), dpi=150)

x = np.arange(len(norm_metrics))
width = 0.8 / len(SOURCE_ORDER)

for i, source in enumerate(SOURCE_ORDER):
    offset = (i - len(SOURCE_ORDER) / 2 + 0.5) * width
    ax.bar(
        x + offset, z_medians.loc[source], width,
        color=SOURCE_COLORS.get(source, '#888888'),
        edgecolor='#222222',
        linewidth=0.5,
        label=SOURCE_LABELS.get(source, source),
    )

ax.axhline(y=0, color='#222222', linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels([
    'Certainty', 'Deontic\nModality', 'You-\nDensity', 'Question\nRatio',
    'Leave\nOrientation', 'Therapy\nLanguage', 'Intensifi-\ncation', 'Sentiment'
], fontsize=9)
ax.set_ylabel('Z-scored Median')
ax.legend(loc='upper right', framealpha=0.9, ncol=2)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3, linestyle='--')

plt.tight_layout()
plt.show()

In [ ]:
# Save Figure 5
fig.savefig(FIGURES_DIR / 'fig_advice_normalized_profiles.pdf', bbox_inches='tight', dpi=300)
fig.savefig(FIGURES_DIR / 'fig_advice_normalized_profiles.png', bbox_inches='tight', dpi=300)
print("Saved: fig_advice_normalized_profiles.pdf/png")

## Inter-LLM Agreement: Do Models That Categorized Similarly Also Advise Similarly?

Correlate each LLM pair's metric profiles to see if advice style clusters by provider.

In [ ]:
# For each LLM, compute its median metric profile
profile_metrics = [
    'certainty_ratio', 'modal_ratio', 'you_density', 'question_ratio',
    'imperative_count', 'leave_ratio', 'redflag_count', 'therapy_count',
    'intensification_ratio', 'sentiment_compound',
    'first_person_ratio', 'second_person_ratio',
]

profiles = df.groupby('source')[profile_metrics].median()
profiles = profiles.loc[[s for s in SOURCE_ORDER if s in profiles.index]]

# Pairwise correlation of profiles
profile_corr = profiles.T.corr()

fig, ax = plt.subplots(figsize=(6, 5), dpi=150)

im = ax.imshow(profile_corr.values, cmap='Greys', vmin=-1, vmax=1)
ax.set_xticks(range(len(profile_corr)))
ax.set_yticks(range(len(profile_corr)))
ax.set_xticklabels([SOURCE_LABELS.get(s, s) for s in profile_corr.index],
                    rotation=30, ha='right', fontsize=9)
ax.set_yticklabels([SOURCE_LABELS.get(s, s) for s in profile_corr.index], fontsize=9)

# Annotate
for i in range(len(profile_corr)):
    for j in range(len(profile_corr)):
        val = profile_corr.values[i, j]
        color = 'white' if abs(val) > 0.5 else 'black'
        ax.text(j, i, f"{val:.2f}", ha='center', va='center',
                fontsize=9, color=color)

plt.colorbar(im, ax=ax, label='Pearson $r$', shrink=0.8)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# Save Figure 6
fig.savefig(FIGURES_DIR / 'fig_advice_profile_correlation.pdf', bbox_inches='tight', dpi=300)
fig.savefig(FIGURES_DIR / 'fig_advice_profile_correlation.png', bbox_inches='tight', dpi=300)
print("Saved: fig_advice_profile_correlation.pdf/png")

## Response Length Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4), dpi=150)

for source in SOURCE_ORDER:
    vals = df[df['source'] == source]['n_tokens'].dropna()
    ax.hist(
        vals, bins=50, alpha=0.6, density=True,
        color=SOURCE_COLORS.get(source, '#888888'),
        edgecolor='none',
        label=f"{SOURCE_LABELS.get(source, source)} (med={vals.median():.0f})",
    )

ax.set_xlabel('Response Length (tokens)')
ax.set_ylabel('Density')
ax.legend(loc='upper right', framealpha=0.9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_xlim(0, ax.get_xlim()[1])

plt.tight_layout()
plt.show()

In [ ]:
# Save
fig.savefig(FIGURES_DIR / 'fig_advice_length_distribution.pdf', bbox_inches='tight', dpi=300)
fig.savefig(FIGURES_DIR / 'fig_advice_length_distribution.png', bbox_inches='tight', dpi=300)
print("Saved: fig_advice_length_distribution.pdf/png")

## Length-Normalized Metrics

Since LLMs produce longer responses, normalize counts by response length
to check whether differences survive length correction.

In [ ]:
# Normalize count metrics per 100 tokens
count_metrics = [
    'hedge_count', 'booster_count', 'deontic_count', 'epistemic_count',
    'imperative_count', 'conditional_count',
    'leave_count', 'stay_count', 'redflag_count', 'therapy_count',
    'intensifier_count', 'downtoner_count',
]

df_norm = df.copy()
for col in count_metrics:
    df_norm[f'{col}_per100'] = df_norm[col] / df_norm['n_tokens'] * 100

# Compare normalized rates
norm_summary_cols = [f'{c}_per100' for c in count_metrics]
norm_summary = df_norm.groupby('source')[norm_summary_cols].median()
norm_summary = norm_summary.loc[[s for s in SOURCE_ORDER if s in norm_summary.index]]
print("Normalized counts (per 100 tokens), medians:")
print(norm_summary.round(3).to_string())

In [ ]:
# Key normalized metrics comparison
fig, axes = plt.subplots(2, 3, figsize=(14, 8), dpi=150)

norm_panels = [
    ('hedge_count_per100', 'Hedges per 100 tokens'),
    ('booster_count_per100', 'Boosters per 100 tokens'),
    ('deontic_count_per100', 'Deontic modals per 100 tokens'),
    ('imperative_count_per100', 'Imperatives per 100 tokens'),
    ('therapy_count_per100', 'Therapy language per 100 tokens'),
    ('redflag_count_per100', 'Red flags per 100 tokens'),
]

panel_labels = ['(a)', '(b)', '(c)', '(d)', '(e)', '(f)']

for idx, (metric, ylabel) in enumerate(norm_panels):
    ax = axes.flat[idx]
    
    data_groups = [df_norm[df_norm['source'] == s][metric].dropna() for s in SOURCE_ORDER]
    
    bp = ax.boxplot(
        data_groups, positions=range(len(SOURCE_ORDER)), widths=0.6,
        patch_artist=True, showfliers=False,
        medianprops=dict(color='black', linewidth=1.5),
        whiskerprops=dict(color='#333333'),
        capprops=dict(color='#333333'),
    )
    
    for patch, source in zip(bp['boxes'], SOURCE_ORDER):
        patch.set_facecolor(SOURCE_COLORS.get(source, '#888888'))
        patch.set_edgecolor('#222222')
        patch.set_linewidth(0.5)
    
    ax.set_xticks(range(len(SOURCE_ORDER)))
    ax.set_xticklabels([SOURCE_LABELS.get(s, s) for s in SOURCE_ORDER],
                        rotation=30, ha='right', fontsize=8)
    ax.set_ylabel(ylabel)
    ax.set_title(panel_labels[idx], fontweight='bold', loc='left')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='y', alpha=0.3, linestyle='--')

plt.tight_layout()
plt.show()

In [ ]:
# Save
fig.savefig(FIGURES_DIR / 'fig_advice_normalized_counts.pdf', bbox_inches='tight', dpi=300)
fig.savefig(FIGURES_DIR / 'fig_advice_normalized_counts.png', bbox_inches='tight', dpi=300)
print("Saved: fig_advice_normalized_counts.pdf/png")

## Summary Table for Paper

In [ ]:
# Generate LaTeX-ready summary table
paper_metrics = {
    'n_tokens': 'Tokens',
    'certainty_ratio': 'Certainty ratio',
    'modal_ratio': 'Deontic ratio',
    'you_density': 'You-density',
    'question_ratio': 'Question ratio',
    'leave_ratio': 'Leave ratio',
    'therapy_count': 'Therapy terms',
    'redflag_count': 'Red flag terms',
    'intensification_ratio': 'Intensification',
    'sentiment_compound': 'Sentiment',
}

table_data = []
for metric_key, metric_label in paper_metrics.items():
    row = {'Metric': metric_label}
    for source in SOURCE_ORDER:
        source_vals = df[df['source'] == source][metric_key].dropna()
        med = source_vals.median()
        iqr = source_vals.quantile(0.75) - source_vals.quantile(0.25)
        row[SOURCE_LABELS.get(source, source)] = f"{med:.2f} ({iqr:.2f})"
    table_data.append(row)

table_df = pd.DataFrame(table_data)
print(table_df.to_string(index=False))
print("\n\nLaTeX:")
print(table_df.to_latex(index=False, escape=True))

---
## Norm-Enforcement-Stratified Analysis

The interaction analysis reveals that different topics trigger different levels of community norm
enforcement (measured by OP downvote rate). We test whether LLM-human divergence is larger on
**strict topics** (where the community has clear normative answers) versus **lenient topics**
(where the community treats the situation as genuinely uncertain).

**Hypothesis:** LLMs will diverge most from humans on high-enforcement topics, because these
represent the strongest community-specific norms — precisely the kind of culturally-contingent
"doxa" that LLMs fail to reproduce.

In [ ]:
# Load topic assignments and compute norm enforcement from OP replies

# 1. Load topic assignments (primary_topic per post)
topic_files = sorted(Path('../data/topic_assignment').glob('posts_with_topics_*_094411.parquet'))
if not topic_files:
    topic_files = sorted(Path('../data/topic_assignment').glob('posts_with_topics_*.parquet'))
topics_df = pd.read_parquet(topic_files[-1])[['post_id', 'primary_topic']]
print(f"Topic assignments: {len(topics_df)} posts, {topics_df['primary_topic'].nunique()} topics")

# 2. Merge topics with advice metrics
df_topics = df.merge(topics_df, on='post_id', how='inner')
print(f"Metrics rows with topic assignment: {len(df_topics)} / {len(df)}")

# 3. Compute norm enforcement from comments (OP downvote rate per topic)
comments = pd.read_parquet('../data/r_relationship_advice_comments_cleaned.parquet')
op_replies = comments[
    (comments['is_op'] == True) &
    (comments['is_top_level'] == False)
].copy()
op_replies['post_id'] = op_replies['link_id'].str.replace('t3_', '', regex=False)

# Merge OP replies with topics
op_with_topics = op_replies.merge(topics_df, on='post_id', how='inner')
op_with_topics = op_with_topics[op_with_topics['primary_topic'].notna()]

# Compute per-topic norm enforcement (OP downvote rate)
topic_enforcement = op_with_topics.groupby('primary_topic').agg(
    n_op_replies=('score', 'count'),
    downvote_rate=('score', lambda x: (x < 0).mean() * 100),
).query('n_op_replies >= 50').sort_values('downvote_rate', ascending=False)

print(f"\nTopics with 50+ OP replies: {len(topic_enforcement)}")

# Classify into quartiles: strict (top 25%) vs lenient (bottom 25%)
q75 = topic_enforcement['downvote_rate'].quantile(0.75)
q25 = topic_enforcement['downvote_rate'].quantile(0.25)

strict_topics = set(topic_enforcement[topic_enforcement['downvote_rate'] >= q75].index)
lenient_topics = set(topic_enforcement[topic_enforcement['downvote_rate'] <= q25].index)

print(f"Strict topics (top quartile, downvote >= {q75:.1f}%): {len(strict_topics)}")
print(f"Lenient topics (bottom quartile, downvote <= {q25:.1f}%): {len(lenient_topics)}")
print(f"\nStrictest 5: {list(topic_enforcement.head(5).index)}")
print(f"Most lenient 5: {list(topic_enforcement.tail(5).index)}")


In [ ]:
# Compare LLM-human divergence on strict vs lenient topics
# Cohen's d for each metric, split by enforcement level

df_strict = df_topics[df_topics['primary_topic'].isin(strict_topics)]
df_lenient = df_topics[df_topics['primary_topic'].isin(lenient_topics)]

print(f"Strict topic rows: {len(df_strict)} ({df_strict['source'].value_counts().to_dict()})")
print(f"Lenient topic rows: {len(df_lenient)} ({df_lenient['source'].value_counts().to_dict()})")

key_metrics = [
    ('certainty_ratio', 'Certainty'),
    ('modal_ratio', 'Deontic Modality'),
    ('you_density', 'You-Density'),
    ('leave_ratio', 'Leave Orientation'),
    ('therapy_count', 'Therapy Language'),
    ('redflag_count', 'Red Flag Language'),
    ('sentiment_compound', 'Sentiment'),
    ('intensification_ratio', 'Intensification'),
]

llm_sources = [s for s in SOURCE_ORDER if s != 'human']

# Compute effect sizes for strict and lenient subsets
enforcement_effects = []
for metric, label in key_metrics:
    for source in llm_sources:
        for subset_name, subset_df in [('strict', df_strict), ('lenient', df_lenient)]:
            human_vals = subset_df[subset_df['source'] == 'human'][metric].dropna()
            llm_vals = subset_df[subset_df['source'] == source][metric].dropna()
            if len(human_vals) >= 10 and len(llm_vals) >= 10:
                d = cohens_d(llm_vals, human_vals)
            else:
                d = np.nan
            enforcement_effects.append({
                'metric': label, 'source': source,
                'enforcement': subset_name, 'd': d,
            })

enf_df = pd.DataFrame(enforcement_effects)

# Summary: average |d| by enforcement level
print("\nMean |Cohen's d| by enforcement level (averaged across models):")
summary = enf_df.groupby(['enforcement', 'metric'])['d'].apply(lambda x: x.abs().mean()).unstack('enforcement')
summary['diff'] = summary['strict'] - summary['lenient']
print(summary.round(3).to_string())
print(f"\nOverall: strict={enf_df[enf_df['enforcement']=='strict']['d'].abs().mean():.3f}, "
      f"lenient={enf_df[enf_df['enforcement']=='lenient']['d'].abs().mean():.3f}")


In [ ]:
# Figure: Effect size comparison — strict vs lenient topics
fig, axes = plt.subplots(1, 2, figsize=(14, 6), dpi=150, sharey=True)

for ax_idx, (enforcement_label, enforcement_name) in enumerate([('Strict Topics\n(top quartile norm enforcement)', 'strict'),
                                                                  ('Lenient Topics\n(bottom quartile norm enforcement)', 'lenient')]):
    ax = axes[ax_idx]
    subset = enf_df[enf_df['enforcement'] == enforcement_name]
    pivot = subset.pivot(index='metric', columns='source', values='d')
    pivot = pivot[[s for s in llm_sources if s in pivot.columns]]
    
    # Sort by average absolute effect
    metric_order = pivot.abs().mean(axis=1).sort_values(ascending=True).index
    pivot = pivot.loc[metric_order]
    
    y_positions = np.arange(len(pivot))
    n_sources = len(pivot.columns)
    bar_height = 0.8 / n_sources
    
    for i, source in enumerate(pivot.columns):
        offset = (i - n_sources / 2 + 0.5) * bar_height
        ax.barh(
            y_positions + offset, pivot[source], height=bar_height,
            color=SOURCE_COLORS.get(source, '#888888'),
            edgecolor='#222222', linewidth=0.5,
            label=SOURCE_LABELS.get(source, source) if ax_idx == 0 else None,
        )
    
    ax.axvline(x=0, color='#222222', linewidth=0.8)
    for threshold in [-0.5, -0.2, 0.2, 0.5]:
        ax.axvline(x=threshold, color='#cccccc', linewidth=0.5, linestyle=':')
    
    ax.set_yticks(y_positions)
    ax.set_yticklabels(pivot.index if ax_idx == 0 else [])
    ax.set_xlabel("Cohen's $d$ (LLM $-$ Human)")
    ax.set_title(f"({'a' if ax_idx == 0 else 'b'})  {enforcement_label}", fontweight='bold', loc='left', fontsize=10)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='x', alpha=0.3, linestyle='--')

axes[0].legend(loc='lower right', framealpha=0.9, fontsize=8)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_advice_enforcement_stratified.pdf', bbox_inches='tight', dpi=300)
plt.savefig(FIGURES_DIR / 'fig_advice_enforcement_stratified.png', bbox_inches='tight', dpi=300)
print('Saved: fig_advice_enforcement_stratified.pdf/png')
plt.show()


---
## Topic-Level Leave/Stay and Hedging on Strict Topics

On high-enforcement topics (abuse, infidelity, safety), the community has clear normative
positions (usually: leave). Do LLMs reproduce this decisiveness, or do they hedge more
precisely where humans are most certain? This tests whether LLMs' "both-sides" tendency
clashes with community monodoxy.


In [ ]:
# Leave/stay ratio and certainty on strict vs lenient topics, by source
focus_metrics = ['leave_ratio', 'certainty_ratio', 'modal_ratio', 'therapy_count']
focus_labels = ['Leave Ratio', 'Certainty Ratio', 'Deontic Ratio', 'Therapy Terms']

# Per-source medians on strict vs lenient
print("Medians by source and enforcement level:")
print("=" * 80)
for metric, label in zip(focus_metrics, focus_labels):
    print(f"\n{label}:")
    for enforcement, subset_df in [('STRICT', df_strict), ('LENIENT', df_lenient)]:
        vals = subset_df.groupby('source')[metric].median()
        vals = vals.reindex([s for s in SOURCE_ORDER if s in vals.index])
        parts = [f"{SOURCE_LABELS.get(s, s)}: {v:.3f}" for s, v in vals.items()]
        print(f"  {enforcement:>8}: {', '.join(parts)}")


In [ ]:
# Figure: Leave ratio and certainty on strict topics (where it matters most)
fig, axes = plt.subplots(1, 4, figsize=(16, 5), dpi=150)

for idx, (metric, label) in enumerate(zip(focus_metrics, focus_labels)):
    ax = axes[idx]
    
    # Grouped bars: strict vs lenient, by source
    x = np.arange(len(SOURCE_ORDER))
    width = 0.35
    
    strict_medians = [df_strict[df_strict['source'] == s][metric].median() for s in SOURCE_ORDER]
    lenient_medians = [df_lenient[df_lenient['source'] == s][metric].median() for s in SOURCE_ORDER]
    
    ax.bar(x - width/2, strict_medians, width,
           color='#333333', edgecolor='#111111', linewidth=0.5, label='Strict topics')
    ax.bar(x + width/2, lenient_medians, width,
           color='#aaaaaa', edgecolor='#555555', linewidth=0.5, label='Lenient topics')
    
    ax.set_xticks(x)
    ax.set_xticklabels([SOURCE_LABELS.get(s, s) for s in SOURCE_ORDER],
                        rotation=35, ha='right', fontsize=8)
    ax.set_ylabel(label)
    ax.set_title(f"({'abcd'[idx]})", fontweight='bold', loc='left')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    if idx == 0:
        ax.legend(fontsize=8, loc='upper right')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_advice_strict_vs_lenient.pdf', bbox_inches='tight', dpi=300)
plt.savefig(FIGURES_DIR / 'fig_advice_strict_vs_lenient.png', bbox_inches='tight', dpi=300)
print('Saved: fig_advice_strict_vs_lenient.pdf/png')
plt.show()


---
## Model Bias x Advice Style

The paper's core hypothesis: models that over-attend to a topic during discovery
(high bias score) will also produce distinctive advice on that topic. We test
whether discovery bias predicts advice divergence from human baseline.

For each model-topic pair, we compute:
- **Discovery bias**: `P(t|m) / P(t)` from the bias matrix
- **Advice divergence**: absolute Cohen's d from human on key metrics for that topic

If bias predicts divergence, it connects categorization to normative behavior.


In [ ]:
# Load discovery bias matrix
bias_matrix = pd.read_csv('../data/model_topic_bias/model_topic_bias_matrix.csv', index_col=0)
print(f"Bias matrix: {bias_matrix.shape[0]} models x {bias_matrix.shape[1]} topics")

# Map discovery model names to advice source names
# Only 4 models have both discovery and advice data
discovery_to_advice = {
    'Gemini 2.5 FL': 'gemini',
    'DeepSeek v3.2': 'deepseek',
    'Ministral 8B': 'ministral',
    'GPT-4.1-nano': 'gpt_nano',
}

# For each model-topic pair: compute advice divergence from human
divergence_metrics = ['certainty_ratio', 'modal_ratio', 'you_density',
                      'leave_ratio', 'therapy_count', 'sentiment_compound']

bias_divergence_pairs = []
for discovery_name, advice_source in discovery_to_advice.items():
    if discovery_name not in bias_matrix.index:
        continue
    if advice_source not in df_topics['source'].unique():
        continue
    
    for topic in bias_matrix.columns:
        bias_score = bias_matrix.loc[discovery_name, topic]
        
        # Get advice data for this model-topic pair
        topic_human = df_topics[(df_topics['source'] == 'human') &
                                (df_topics['primary_topic'] == topic)]
        topic_llm = df_topics[(df_topics['source'] == advice_source) &
                              (df_topics['primary_topic'] == topic)]
        
        if len(topic_human) < 5 or len(topic_llm) < 5:
            continue
        
        # Average absolute Cohen's d across metrics
        ds = []
        for metric in divergence_metrics:
            h = topic_human[metric].dropna()
            l = topic_llm[metric].dropna()
            if len(h) >= 5 and len(l) >= 5:
                ds.append(abs(cohens_d(l, h)))
        
        if ds:
            bias_divergence_pairs.append({
                'model': advice_source,
                'model_display': SOURCE_LABELS.get(advice_source, advice_source),
                'topic': topic,
                'bias_score': bias_score,
                'mean_divergence': np.mean(ds),
                'n_posts': len(topic_llm),
            })

bias_div_df = pd.DataFrame(bias_divergence_pairs)
print(f"\nModel-topic pairs with data: {len(bias_div_df)}")
print(f"Per model: {bias_div_df['model'].value_counts().to_dict()}")

# Overall correlation: does bias predict divergence?
from scipy.stats import pearsonr, spearmanr
if len(bias_div_df) > 10:
    r_p, p_p = pearsonr(bias_div_df['bias_score'], bias_div_df['mean_divergence'])
    r_s, p_s = spearmanr(bias_div_df['bias_score'], bias_div_df['mean_divergence'])
    print(f"\nBias -> Divergence correlation:")
    print(f"  Pearson:  r={r_p:.3f}, p={p_p:.4f}")
    print(f"  Spearman: r={r_s:.3f}, p={p_s:.4f}")
    
    # Per-model correlations
    print("\nPer-model:")
    for model in bias_div_df['model'].unique():
        m_df = bias_div_df[bias_div_df['model'] == model]
        if len(m_df) >= 5:
            r, p = spearmanr(m_df['bias_score'], m_df['mean_divergence'])
            print(f"  {SOURCE_LABELS.get(model, model):<15}: r={r:+.3f}, p={p:.4f} (n={len(m_df)})")


In [ ]:
# Figure: Discovery bias vs advice divergence
fig, axes = plt.subplots(1, 2, figsize=(12, 5), dpi=150)

# (a) Scatter: all model-topic pairs
ax = axes[0]
for model in bias_div_df['model'].unique():
    m_df = bias_div_df[bias_div_df['model'] == model]
    ax.scatter(
        m_df['bias_score'], m_df['mean_divergence'],
        s=m_df['n_posts'] * 2,
        color=SOURCE_COLORS.get(model, '#888888'),
        edgecolors='#222222', linewidth=0.3,
        alpha=0.6, label=SOURCE_LABELS.get(model, model),
    )

# Regression line (overall)
if len(bias_div_df) > 10:
    z = np.polyfit(bias_div_df['bias_score'], bias_div_df['mean_divergence'], 1)
    x_line = np.linspace(bias_div_df['bias_score'].min(), bias_div_df['bias_score'].max(), 50)
    ax.plot(x_line, np.polyval(z, x_line), '--', color='#333333', linewidth=1.2, alpha=0.8)

ax.set_xlabel('Discovery Bias Score (lift)')
ax.set_ylabel('Mean |Cohen\'s $d$| from Human')
ax.legend(fontsize=8, loc='upper right')
ax.set_title('(a)', fontweight='bold', loc='left')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# (b) Per-model correlation summary (bar chart)
ax = axes[1]
model_corrs = []
for model in [s for s in SOURCE_ORDER if s != 'human']:
    m_df = bias_div_df[bias_div_df['model'] == model]
    if len(m_df) >= 5:
        r, p = spearmanr(m_df['bias_score'], m_df['mean_divergence'])
        model_corrs.append({'model': model, 'r': r, 'p': p})

if model_corrs:
    corr_df = pd.DataFrame(model_corrs)
    x_pos = np.arange(len(corr_df))
    bars = ax.bar(
        x_pos, corr_df['r'],
        color=[SOURCE_COLORS.get(m, '#888888') for m in corr_df['model']],
        edgecolor='#222222', linewidth=0.5,
    )
    # Mark significance
    for i, row in corr_df.iterrows():
        if row['p'] < 0.05:
            ax.text(i, row['r'] + 0.02 * np.sign(row['r']),
                    '*', ha='center', fontsize=14, fontweight='bold')
    
    ax.axhline(y=0, color='#222222', linewidth=0.8)
    ax.set_xticks(x_pos)
    ax.set_xticklabels([SOURCE_LABELS.get(m, m) for m in corr_df['model']],
                        rotation=30, ha='right', fontsize=9)
    ax.set_ylabel('Spearman $r$ (bias vs divergence)')
    ax.set_title('(b)', fontweight='bold', loc='left')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_advice_bias_divergence.pdf', bbox_inches='tight', dpi=300)
plt.savefig(FIGURES_DIR / 'fig_advice_bias_divergence.png', bbox_inches='tight', dpi=300)
print('Saved: fig_advice_bias_divergence.pdf/png')
plt.show()


---
## Therapy Language Distribution by Topic

Do LLMs apply therapeutic framing uniformly across topics, or do they modulate
by problem type like humans? If therapy language is evenly distributed in LLM
advice but concentrated on specific topics in human advice, that reveals LLMs
applying a generic therapeutic frame regardless of context.


In [ ]:
# Compute per-topic therapy language rate for each source
# Normalize by token count to control for response length
df_topics_norm = df_topics.copy()
df_topics_norm['therapy_per100'] = df_topics_norm['therapy_count'] / df_topics_norm['n_tokens'] * 100

# Per-topic, per-source median therapy rate
therapy_by_topic = df_topics_norm.groupby(['primary_topic', 'source'])['therapy_per100'].median().unstack('source')
therapy_by_topic = therapy_by_topic[[s for s in SOURCE_ORDER if s in therapy_by_topic.columns]]

# Only topics with enough data
topic_counts = df_topics.groupby('primary_topic')['source'].count()
valid_topics = topic_counts[topic_counts >= 20].index
therapy_by_topic = therapy_by_topic.loc[therapy_by_topic.index.isin(valid_topics)]

# Coefficient of variation across topics for each source
# Higher CV = more topic-specific (concentrated on certain topics)
# Lower CV = more uniform (applied generically)
print("Therapy language distribution across topics (CV = std/mean):")
print("Higher CV = more topic-specific, lower = more uniform/generic\n")
for source in SOURCE_ORDER:
    if source in therapy_by_topic.columns:
        vals = therapy_by_topic[source].dropna()
        cv = vals.std() / vals.mean() if vals.mean() > 0 else 0
        print(f"  {SOURCE_LABELS.get(source, source):<15}: CV={cv:.3f} (mean={vals.mean():.3f}, std={vals.std():.3f})")

# Top 5 topics for therapy language by source
print("\nTop 5 therapy-heavy topics per source:")
for source in SOURCE_ORDER:
    if source in therapy_by_topic.columns:
        top5 = therapy_by_topic[source].nlargest(5)
        print(f"  {SOURCE_LABELS.get(source, source)}:")
        for topic, val in top5.items():
            print(f"    {topic}: {val:.3f}")


In [ ]:
# Figure: Therapy language concentration — human vs LLM
fig, axes = plt.subplots(1, 2, figsize=(14, 6), dpi=150)

# (a) Sorted therapy rates: human vs average LLM
ax = axes[0]
if 'human' in therapy_by_topic.columns:
    human_sorted = therapy_by_topic['human'].dropna().sort_values(ascending=False)
    llm_cols = [s for s in llm_sources if s in therapy_by_topic.columns]
    llm_avg = therapy_by_topic[llm_cols].mean(axis=1).reindex(human_sorted.index)
    
    x = np.arange(len(human_sorted))
    ax.bar(x - 0.2, human_sorted.values, 0.4, color='#222222', label='Human', edgecolor='#111111', linewidth=0.3)
    ax.bar(x + 0.2, llm_avg.values, 0.4, color='#999999', label='LLM avg', edgecolor='#555555', linewidth=0.3)
    
    # Only label every 5th topic
    ax.set_xticks(x[::5])
    ax.set_xticklabels([t.replace('_', '\n') for t in human_sorted.index[::5]],
                        fontsize=6, rotation=45, ha='right')
    ax.set_ylabel('Therapy terms per 100 tokens')
    ax.set_title('(a)', fontweight='bold', loc='left')
    ax.legend(fontsize=8)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

# (b) Scatter: human therapy rate vs LLM therapy rate per topic
ax = axes[1]
if 'human' in therapy_by_topic.columns and llm_cols:
    for source in llm_cols:
        valid = therapy_by_topic[['human', source]].dropna()
        ax.scatter(valid['human'], valid[source],
                   color=SOURCE_COLORS.get(source, '#888888'),
                   edgecolors='#222222', linewidth=0.3,
                   alpha=0.6, s=30,
                   label=SOURCE_LABELS.get(source, source))
    
    # Identity line
    lims = [0, max(therapy_by_topic.max().max(), 1)]
    ax.plot(lims, lims, '--', color='#666666', linewidth=0.8, alpha=0.7, label='y=x')
    ax.set_xlabel('Human therapy rate (per 100 tokens)')
    ax.set_ylabel('LLM therapy rate (per 100 tokens)')
    ax.set_title('(b)', fontweight='bold', loc='left')
    ax.legend(fontsize=7, loc='upper left')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_advice_therapy_distribution.pdf', bbox_inches='tight', dpi=300)
plt.savefig(FIGURES_DIR / 'fig_advice_therapy_distribution.png', bbox_inches='tight', dpi=300)
print('Saved: fig_advice_therapy_distribution.pdf/png')
plt.show()
